In [ ]:
# Dependency install cell (ultimate one-shot setup for Kani + Qwen HF on SageMaker)
# Run once in a fresh kernel, then restart kernel.

import sys
import subprocess

py = sys.executable

def run(cmd):
  print(">", " ".join(cmd))
  subprocess.check_call(cmd)

# 1) Clean up known conflicting packages
run([py, "-m", "pip", "uninstall", "-y", "torchvision", "transformers", "tokenizers", "kani"])

# 2) Install a compatible set for kani HuggingEngine
run([py, "-m", "pip", "install", "--no-cache-dir", "-U",
   "kani[huggingface]==1.8.0",
   "transformers==4.51.3",
   "tokenizers==0.21.1",
   "accelerate>=0.30.0",
   "huggingface_hub>=0.30.0",
   "sentencepiece",
   "safetensors",
   "protobuf",
   "nest_asyncio",
   "python-dotenv",
   "tqdm"])

# 3) Ensure torch exists (SageMaker usually already has it)
try:
  import torch  # noqa: F401
  print("torch already available")
except Exception:
  run([py, "-m", "pip", "install", "--no-cache-dir", "-U", "torch"])

print("\nInstall complete. IMPORTANT: Restart kernel now, then run Cell 1.")

In [ ]:
# Quick test cell (run after kernel restart)

import importlib.metadata as md

print("python:", __import__("sys").executable)
for p in ["kani", "torch", "transformers", "accelerate", "tokenizers", "huggingface_hub"]:
  try:
      print(f"{p}: {md.version(p)}")
  except Exception:
      print(f"{p}: NOT INSTALLED")

from kani import Kani
from kani.engines.huggingface import HuggingEngine

print("Kani import OK")
print("HuggingEngine import OK")

In [ ]:
# Cell 1: Setup imports, paths, and Qwen2.5 Kani runtime (independent)
from pathlib import Path
import os
import re
import json
import time
import asyncio
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from kani import Kani
from kani.engines.huggingface import HuggingEngine

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Put heavy caches on NVMe (recommended in SageMaker)
NVME_CACHE_ROOT = Path('/mnt/sagemaker-nvme/formalizer_cache')
HF_HOME_DIR = NVME_CACHE_ROOT / 'hf_home'
HF_HUB_CACHE_DIR = NVME_CACHE_ROOT / 'hf_hub'
TRANSFORMERS_CACHE_DIR = NVME_CACHE_ROOT / 'transformers'
TORCH_HOME_DIR = NVME_CACHE_ROOT / 'torch'
XDG_CACHE_DIR = NVME_CACHE_ROOT / 'xdg_cache'

for d in [
  NVME_CACHE_ROOT,
  HF_HOME_DIR,
  HF_HUB_CACHE_DIR,
  TRANSFORMERS_CACHE_DIR,
  TORCH_HOME_DIR,
  XDG_CACHE_DIR,
]:
  d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HUB_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(TRANSFORMERS_CACHE_DIR)
os.environ['TORCH_HOME'] = str(TORCH_HOME_DIR)
os.environ['XDG_CACHE_HOME'] = str(XDG_CACHE_DIR)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DATA_DIR = PROJECT_ROOT / 'data'
PROMPT_PATH = PROJECT_ROOT / 'prompts' / 'only_pddl_instruction_unraveling.txt'
DOMAIN_PDDL_PATH = DATA_DIR / 'pddl' / 'domain.pddl'
DESC_DIR = DATA_DIR / 'descriptions_unraveling'

OUTPUT_ROOT = PROJECT_ROOT / 'output' / 'direct_pddl_unraveling' / 'qwen2_5_32b_instruct_kani_retry3_unraveling'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path('/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate')

# Model setup (Qwen2.5 Instruct via local HuggingFace + Kani)
MODEL_NAME = 'Qwen2.5-32B-Instruct'
HF_MODEL_ID = f'Qwen/{MODEL_NAME}'

QWEN_ENGINE = HuggingEngine(
  model_id=HF_MODEL_ID,
  model_load_kwargs={
      'device_map': 'auto',
      'cache_dir': str(TRANSFORMERS_CACHE_DIR),
  },
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('VAL exists:', VAL_PATH.exists())
print('Domain PDDL exists:', DOMAIN_PDDL_PATH.exists())
print('Prompt exists:', PROMPT_PATH.exists())
print('Model:', HF_MODEL_ID)
print('NVME cache root:', NVME_CACHE_ROOT)

In [ ]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 101)]  # p01..p100

PROMPT_PREFIX = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

In [ ]:
# Cell 3: Load NL descriptions for one problem ID
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain NL file: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem NL file: {problem_path}")

  return domain_path.read_text(), problem_path.read_text()